In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import numpy as np

import sys
sys.path.append("..")

from analysis import *

In [ ]:
df = pd.read_csv("single/version_2/predictions_single.csv")
df = df.query("boosted_h_candidate_score >= 0.1 & boosted_v_candidate_score >= 0.1")

In [ ]:
df_sig = df[df["label"] == 1]
df_bkg = df[df["label"] == 0]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

df_sig.boosted_h_candidate_score.hist(bins=50, alpha=0.5, label="Signal", ax=ax, density=True)
df_bkg.boosted_h_candidate_score.hist(bins=50, alpha=0.5, label="Background", ax=ax, density=True)
plt.xlabel("Boosted H Candidate Score")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
df_sig.boosted_v_candidate_score.hist(bins=50, alpha=0.5, label="Signal", ax=ax, density=True)
df_bkg.boosted_v_candidate_score.hist(bins=50, alpha=0.5, label="Background", ax=ax, density=True)
plt.xlabel("Boosted V Candidate Score")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
sig, (cut_var1_best, cut_var2_best, _, cut_dnn_best, vbs_cut_best) = optimize_cuts(
    df_sig,
    df_bkg,
    var1_col="boosted_h_candidate_score",
    var2_col="boosted_v_candidate_score",
    disco1_col="dnn_score",
    disco2_col="vbs_score",
    cut_var1_list=np.linspace(0.1, 1.0, 46),
    cut_var2_list=np.linspace(0.1, 1.0, 46),
    cut_disco1_list=np.linspace(0., 1.0, 51),
    cut_disco2_list=np.linspace(0., 1.0, 51),
    combine_method="and"
)

In [ ]:
print(f"Optimal cuts: var1 > {cut_var1_best}, var2 > {cut_var2_best}, dnn_score > {cut_dnn_best}, vbs_score > {vbs_cut_best}")
print("Significance at optimal cuts:", sig)

In [ ]:
def get_ABCD_regions(df):
    df = df.query("boosted_h_candidate_score >= @cut_var1_best & boosted_v_candidate_score >= @cut_var2_best")
    A = df[(df["dnn_score"] >= cut_dnn_best) & (df["vbs_score"] >= vbs_cut_best)].weight.sum()
    B = df[(df["dnn_score"] >= cut_dnn_best) & (df["vbs_score"] < vbs_cut_best)].weight.sum()
    C = df[(df["dnn_score"] < cut_dnn_best) & (df["vbs_score"] >= vbs_cut_best)].weight.sum()
    D = df[(df["dnn_score"] < cut_dnn_best) & (df["vbs_score"] < vbs_cut_best)].weight.sum()
    return A, B, C, D

In [ ]:
a, b, c, d = get_ABCD_regions(df_sig)

In [ ]:
e, f, g, h = get_ABCD_regions(df_bkg)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.histplot(data=df_bkg.query("boosted_h_candidate_score >= @cut_var1_best & boosted_v_candidate_score >= @cut_var2_best"), x="dnn_score", y="vbs_score", weights="weight", bins=100, color='blue', cbar=True, ax=ax)

# Add lines to mark ABCD regions
ax.axvline(x=cut_dnn_best, color='red', linestyle='--', linewidth=2, label=f'dnn_score = {cut_dnn_best}')
ax.axhline(y=vbs_cut_best, color='red', linestyle='--', linewidth=2, label=f'vbs_score = {vbs_cut_best}')

# ax.legend()
ax.set_title('Background: ABCD Regions')
plt.show()

In [ ]:
# profile plot of bkg in dnn_score vs vbs_score, with color representing the mean weight in each bin
fig, ax = plt.subplots(figsize=(10, 8))
sns.histplot(data=df_bkg.query("boosted_h_candidate_score >= @cut_var1_best & boosted_v_candidate_score >= @cut_var2_best"), x="dnn_score", y="vbs_score", weights="weight", bins=100, color='blue', cbar=True, ax=ax)

# profile of median of vbs_score in bins of dnn_score
bin_means = df_bkg.query("boosted_h_candidate_score >= @cut_var1_best & boosted_v_candidate_score >= @cut_var2_best").groupby(pd.cut(df_bkg["dnn_score"], bins=20))["vbs_score"].mean()
bin_centers = [interval.mid for interval in bin_means.index.categories]
ax.plot(bin_centers, bin_means.values, color='red', marker='o', label='Median vbs_score')
ax.legend()
ax.set_title('Background: dnn_score vs vbs_score with Median Profile')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.histplot(data=df_sig.query("boosted_h_candidate_score >= @cut_var1_best & boosted_v_candidate_score >= @cut_var2_best"), x="dnn_score", y="vbs_score", weights="weight", bins=100, color='red', cbar=True, ax=ax)

# Add lines to mark ABCD regions
ax.axvline(x=cut_dnn_best, color='black', linestyle='--', linewidth=2, label=f'dnn_score = {cut_dnn_best}')
ax.axhline(y=vbs_cut_best, color='black', linestyle='--', linewidth=2, label=f'vbs_score = {vbs_cut_best}')

ax.set_title('Signal: ABCD Regions')
plt.show()

In [ ]:
a, b, c, d

In [ ]:
e, f, g, h

In [ ]:
def get_bkg_A(df, cut_var1_best, cut_var2_best, cut_dnn_best=cut_dnn_best, vbs_cut_best=vbs_cut_best):
    df = df.query("boosted_h_candidate_score >= @cut_var1_best & boosted_v_candidate_score >= @cut_var2_best")
    A = df[(df["dnn_score"] >= cut_dnn_best) & (df["vbs_score"] >= vbs_cut_best)].weight.sum()
    B = df[(df["dnn_score"] >= cut_dnn_best) & (df["vbs_score"] < vbs_cut_best)].weight.sum()
    C = df[(df["dnn_score"] < cut_dnn_best) & (df["vbs_score"] >= vbs_cut_best)].weight.sum()
    D = df[(df["dnn_score"] < cut_dnn_best) & (df["vbs_score"] < vbs_cut_best)].weight.sum()
    return A, B, C, D

bkg_A, bkg_B, bkg_C, bkg_D, bkg_A_pred = [], [], [], [], []

h_bins = np.linspace(0, 1, 21)
for h_cut in h_bins:
    yeld = get_bkg_A(df_bkg, h_cut, cut_var2_best, cut_dnn_best, vbs_cut_best)
    bkg_A.append(yeld[0])
    bkg_B.append(yeld[1])
    bkg_C.append(yeld[2])
    bkg_D.append(yeld[3])
    bkg_A_pred.append(yeld[1] * yeld[2] / yeld[3] if yeld[3] > 0 else 0)

plt.plot(h_bins, bkg_A, marker='o', label='Region A')
plt.plot(h_bins, bkg_B, marker='o', label='Region B')
plt.plot(h_bins, bkg_C, marker='o', label='Region C')
plt.plot(h_bins, bkg_D, marker='o', label='Region D')
plt.plot(h_bins, bkg_A_pred, marker='x', label='Predicted A (B*C/D)')
plt.legend()
plt.yscale('log')


In [ ]:
df_data = pd.read_csv("single/version_2/predictions_single_data.csv")
df_data = df.query("boosted_h_candidate_score >= 0.1 & boosted_v_candidate_score >= 0.1")

In [ ]:
_, j, k, l = get_ABCD_regions(df_data)
_ = -1 # blind

In [ ]:
def get_yields(df, cut_var1, cut_var2, cut_dnn_best=cut_dnn_best, vbs_cut_best=vbs_cut_best):
    df = df.query("boosted_h_candidate_score >= @cut_var1 & boosted_v_candidate_score >= @cut_var2")
    A = df[(df["dnn_score"] >= cut_dnn_best) & (df["vbs_score"] >= vbs_cut_best)].weight.sum()
    B = df[(df["dnn_score"] >= cut_dnn_best) & (df["vbs_score"] < vbs_cut_best)].weight.sum()
    C = df[(df["dnn_score"] < cut_dnn_best) & (df["vbs_score"] >= vbs_cut_best)].weight.sum()
    D = df[(df["dnn_score"] < cut_dnn_best) & (df["vbs_score"] < vbs_cut_best)].weight.sum()
    return A, B, C, D

h_bins = np.linspace(0.1, cut_var1_best // 2, 21)
v_bins = np.linspace(0.1, cut_var2_best // 2, 21)
for v_cut in v_bins:
    bkg_A, bkg_B, bkg_C, bkg_D, bkg_A_pred = [], [], [], [], []
    for h_cut in h_bins:
        yeld = get_yields(df_data, h_cut, v_cut, cut_dnn_best, vbs_cut_best)
        bkg_A.append(yeld[0])
        bkg_B.append(yeld[1])
        bkg_C.append(yeld[2])
        bkg_D.append(yeld[3])
        bkg_A_pred.append(yeld[1] * yeld[2] / yeld[3] if yeld[3] > 0 else 0)

    plt.plot(h_bins, bkg_A, marker='o', label='Region A')
    plt.plot(h_bins, bkg_B, marker='o', label='Region B')
    plt.plot(h_bins, bkg_C, marker='o', label='Region C')
    plt.plot(h_bins, bkg_D, marker='o', label='Region D')
    plt.plot(h_bins, bkg_A_pred, marker='x', label='Predicted A (B*C/D)')
    plt.title(f"V cut = {v_cut:.2f}")
    plt.legend()
    plt.yscale('log')
    plt.show()

